# Evaluation - Résultats et Métriques
Évaluer le modèle et visualiser les résultats

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, roc_auc_score, roc_curve
)
import seaborn as sns
import tensorflow as tf
from tensorflow import keras
from PIL import Image

BASE_DIR = Path('../')
DATA_DIR = BASE_DIR / 'data' / 'raw' / 'chest_xray'
MODEL_DIR = BASE_DIR / 'models'
OUTPUT_DIR = BASE_DIR / 'outputs'

print("📊 Évaluation du modèle...")

### Charger modèle et données

In [ ]:
# Charger modèle
model_path = MODEL_DIR / 'chest_xray_model.h5'
model = keras.models.load_model(model_path)
print(f"✓ Modèle chargé: {model_path}")

# Fonction charger images
def load_images_and_labels(data_dir, split='test', img_size=(224, 224)):
    images = []
    labels = []
    for class_name, label in [('NORMAL', 0), ('PNEUMONIA', 1)]:
        class_dir = data_dir / split / class_name
        if not class_dir.exists():
            continue
        for img_path in class_dir.glob('*.jpeg'):
            try:
                img = Image.open(img_path).convert('L')
                img = img.resize(img_size)
                img_array = np.array(img) / 255.0
                images.append(img_array)
                labels.append(label)
            except:
                pass
    return np.array(images), np.array(labels)

# Charger données test
X_test, y_test = load_images_and_labels(DATA_DIR, 'test')
X_test = np.stack([np.expand_dims(x, -1) for x in X_test])
print(f"✓ Données test: {X_test.shape}")

### Prédictions

In [ ]:
# Prédictions probabilités
y_pred_proba = model.predict(X_test)
y_pred = (y_pred_proba > 0.5).astype(int).flatten()

print(f"✓ Prédictions complétées")
print(f"  Distribution: {sum(y_pred==0)} NORMAL, {sum(y_pred==1)} PNEUMONIA")

### Métriques

In [ ]:
# Calculer métriques
acc = accuracy_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_pred_proba)

print("\n📈 Résultats:")
print(f"  Accuracy:  {acc:.4f}")
print(f"  Precision: {prec:.4f}")
print(f"  Recall:    {rec:.4f}")
print(f"  F1-Score:  {f1:.4f}")
print(f"  ROC-AUC:   {auc:.4f}")

# Sauvegarder
metrics = {
    'Accuracy': acc,
    'Precision': prec,
    'Recall': rec,
    'F1-Score': f1,
    'ROC-AUC': auc
}

pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'metrics.csv', index=False)
print(f"\n✓ Métriques sauvegardées")

### Visualisations

In [ ]:
# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Confusion Matrix
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=axes[0],
            xticklabels=['NORMAL', 'PNEUMONIA'],
            yticklabels=['NORMAL', 'PNEUMONIA'])
axes[0].set_title('Confusion Matrix')
axes[0].set_ylabel('True')
axes[0].set_xlabel('Predicted')

# ROC Curve
fpr, tpr, _ = roc_curve(y_test, y_pred_proba)
axes[1].plot(fpr, tpr, label=f'ROC-AUC: {auc:.3f}', linewidth=2)
axes[1].plot([0, 1], [0, 1], 'k--', linewidth=1, label='Random')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'evaluation_metrics.png', dpi=100, bbox_inches='tight')
plt.show()

print("✓ Graphiques sauvegardés")

### Export des résultats

In [ ]:
# Créer DataFrame des résultats
results = pd.DataFrame({
    'True_Label': y_test,
    'Predicted_Label': y_pred,
    'Probability_PNEUMONIA': y_pred_proba.flatten(),
    'Correct': y_test == y_pred
})

# Ajouter class names
results['True_Class'] = results['True_Label'].map({0: 'NORMAL', 1: 'PNEUMONIA'})
results['Predicted_Class'] = results['Predicted_Label'].map({0: 'NORMAL', 1: 'PNEUMONIA'})

# Sauvegarder
results_path = OUTPUT_DIR / 'predictions.csv'
results.to_csv(results_path, index=False)

print(f"✓ Résultats sauvegardés: {results_path}")
print(f"\nPremières lignes:")
print(results.head(10))

# Résumé
print(f"\n✓ Correctes: {sum(results['Correct'])} / {len(results)} ({100*sum(results['Correct'])/len(results):.1f}%)")